#6.DATA ANALYSIS

#Create Analysis Dataset

In [ ]:
# Create a separate dataset for analysis

df_analysis = df_aggregated.copy()

print("Analysis dataset shape:", df_analysis.shape)

Analysis dataset shape: (225731, 11)


#Clean Medicine Names

In [ ]:
import re

def clean_medicine_name(name):

    name = str(name).lower().strip()

    # Remove strength values
    name = re.sub(
        r'\b\d+(\.\d+)?\s*(mg|mcg|g|kg|ml|l|%|iu)\b',
        '',
        name
    )

    # Remove dosage/form-related words
    dosage_words = [
        "tablet", "tablets",
        "capsule", "capsules",
        "syrup", "suspension",
        "injection", "infusion",
        "cream", "ointment",
        "gel", "drops",
        "solution", "powder",
        "inhaler", "softgel",
        "oral", "dry", "junior"
    ]

    for word in dosage_words:
        name = re.sub(
            r'\b' + word + r'\b',
            '',
            name
        )

    # Remove extra spaces
    name = re.sub(r'\s+', ' ', name).strip()

    return name


# Apply cleaning to medicine names
df_analysis["name_core"] = (
    df_analysis["brand_name"]
    .apply(clean_medicine_name)
)


# Clean primary ingredient names
df_analysis["ingredient_clean"] = (
    df_analysis["primary_ingredient"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Medicine name cleaning completed.")

Medicine name cleaning completed.


#Determine Medicine Type

In [ ]:
# Check whether the cleaned medicine name
# matches the primary ingredient

df_analysis["name_core_matches_ingredient"] = (
    df_analysis["name_core"]
    ==
    df_analysis["ingredient_clean"]
)


# Assign project-defined medicine type

df_analysis["medicine_type"] = "Potential Branded"

df_analysis.loc[
    df_analysis["name_core_matches_ingredient"],
    "medicine_type"
] = "Potential Generic"

In [ ]:
medicine_type_counts = (
    df_analysis["medicine_type"]
    .value_counts()
    .reset_index()
)

medicine_type_counts.columns = [
    "medicine_type",
    "count"
]

print(medicine_type_counts)

       medicine_type   count
0  Potential Branded  225672
1  Potential Generic      59


#Extract Potential Generic Candidates

In [ ]:
generic_candidates = df_analysis[
    df_analysis["medicine_type"] == "Potential Generic"
].copy()

print(
    "Potential Generic Candidates:",
    len(generic_candidates)
)

Potential Generic Candidates: 59


In [ ]:
print(
    generic_candidates[
        [
            "brand_name",
            "primary_ingredient",
            "manufacturer",
            "price_inr",
            "primary_strength",
            "dosage_form",
            "comparison_group",
            "medicine_type"
        ]
    ].to_string(index=False)
)

                         brand_name   primary_ingredient                          manufacturer  price_inr primary_strength dosage_form                    comparison_group     medicine_type
            Adenosine 3mg Injection            Adenosine         Samarth Life Sciences Pvt Ltd     208.02              3mg   injection             adenosine|3mg|injection Potential Generic
           Albendazole 400mg Tablet          Albendazole            Cadila Pharmaceuticals Ltd       9.58            400mg      tablet            albendazole|400mg|tablet Potential Generic
                   Alfuzosin Tablet            Alfuzosin         Care Formulation Labs Pvt Ltd     165.00             10mg      tablet               alfuzosin|10mg|tablet Potential Generic
           Amikacin 250mg Injection             Amikacin          Prem Pharmaceuticals Pvt Ltd      38.70            250mg   injection            amikacin|250mg|injection Potential Generic
  Amoxycillin 125mg Oral Suspension          Amoxycilli

#Identify Comparison Groups

In [ ]:
generic_groups = (
    generic_candidates["comparison_group"]
    .unique()
)

print(
    "Comparison groups containing potential generic candidates:",
    len(generic_groups)
)

Comparison groups containing potential generic candidates: 56


#Separate Other Products in Those Groups

In [ ]:
other_products = df_analysis[
    df_analysis["comparison_group"].isin(generic_groups)
    &
    (df_analysis["medicine_type"] == "Potential Branded")
].copy()

print(
    "Potential Generic Candidates:",
    len(generic_candidates)
)

print(
    "Other Products:",
    len(other_products)
)

Potential Generic Candidates: 59
Other Products: 6993


#Calculate Average Comparison Price

In [ ]:
other_group_average = (
    other_products
    .groupby("comparison_group")["price_inr"]
    .mean()
    .reset_index(name="average_other_price")
)

print(
    other_group_average.head(10)
)

               comparison_group  average_other_price
0       adenosine|3mg|injection           216.296667
1      albendazole|400mg|tablet            30.932322
2         alfuzosin|10mg|tablet           174.442941
3      amikacin|250mg|injection            37.606587
4  amoxycillin|125mg|suspension            42.356129
5       amoxycillin|125mg|syrup            39.952837
6      amoxycillin|125mg|tablet            26.670000
7     amoxycillin|250mg|capsule            54.905263
8      amoxycillin|250mg|tablet           100.263695
9     amoxycillin|500mg|capsule            68.925154


In [ ]:
price_comparison = generic_candidates[
    [
        "comparison_group",
        "brand_name",
        "manufacturer",
        "price_inr"
    ]
].copy()

price_comparison = price_comparison.merge(
    other_group_average,
    on="comparison_group",
    how="left"
)

print(
    price_comparison.head(10)
)

               comparison_group                         brand_name  \
0       adenosine|3mg|injection            Adenosine 3mg Injection   
1      albendazole|400mg|tablet           Albendazole 400mg Tablet   
2         alfuzosin|10mg|tablet                   Alfuzosin Tablet   
3      amikacin|250mg|injection           Amikacin 250mg Injection   
4  amoxycillin|125mg|suspension  Amoxycillin 125mg Oral Suspension   
5       amoxycillin|125mg|syrup            Amoxycillin 125mg Syrup   
6      amoxycillin|125mg|tablet           Amoxycillin 125mg Tablet   
7     amoxycillin|250mg|capsule         Amoxycillin  250mg Capsule   
8      amoxycillin|250mg|tablet           Amoxycillin 250mg Tablet   
9     amoxycillin|500mg|capsule          Amoxycillin 500mg Capsule   

                     manufacturer  price_inr  average_other_price  
0   Samarth Life Sciences Pvt Ltd     208.02           216.296667  
1      Cadila Pharmaceuticals Ltd       9.58            30.932322  
2   Care Formulation Labs

#Calculate Price Difference

In [ ]:
price_comparison["price_difference"] = (
    price_comparison["average_other_price"]
    -
    price_comparison["price_inr"]
)

In [ ]:
price_comparison["price_difference_percent"] = (
    price_comparison["price_difference"]
    /
    price_comparison["average_other_price"]
) * 100

#Generate Final Analysis Table

In [ ]:
price_comparison = price_comparison.sort_values(
    "price_difference_percent",
    ascending=False
).reset_index(drop=True)

print(
    price_comparison.to_string(index=False)
)

                   comparison_group                          brand_name                          manufacturer  price_inr  average_other_price  price_difference  price_difference_percent
             isoniazid|300mg|tablet              Isoniazid 300mg Tablet            Cadila Pharmaceuticals Ltd       1.75            48.201842         46.451842                 96.369433
               aspirin|300mg|tablet                Aspirin 300mg Tablet             Bini Laboratories Pvt Ltd       2.00            22.670000         20.670000                 91.177768
              mesna|200mg|injection                     Mesna Injection                          Zydus Cadila      33.05           315.470000        282.420000                 89.523568
           amoxycillin|250mg|tablet            Amoxycillin 250mg Tablet               Makers Laboratories Ltd      15.00           100.263695         85.263695                 85.039450
          vitamin c|150mg|injection                 Vitamin C Injectio

In [ ]:
final_analysis = price_comparison[
    [
        "comparison_group",
        "brand_name",
        "manufacturer",
        "price_inr",
        "average_other_price",
        "price_difference",
        "price_difference_percent"
    ]
].copy()

print(
    final_analysis.to_string(index=False)
)

                   comparison_group                          brand_name                          manufacturer  price_inr  average_other_price  price_difference  price_difference_percent
             isoniazid|300mg|tablet              Isoniazid 300mg Tablet            Cadila Pharmaceuticals Ltd       1.75            48.201842         46.451842                 96.369433
               aspirin|300mg|tablet                Aspirin 300mg Tablet             Bini Laboratories Pvt Ltd       2.00            22.670000         20.670000                 91.177768
              mesna|200mg|injection                     Mesna Injection                          Zydus Cadila      33.05           315.470000        282.420000                 89.523568
           amoxycillin|250mg|tablet            Amoxycillin 250mg Tablet               Makers Laboratories Ltd      15.00           100.263695         85.263695                 85.039450
          vitamin c|150mg|injection                 Vitamin C Injectio